# 랜덤포레스트 테스트 (초안)

| 항목 | 내용 |
|---|---|
| 데이터 | `data/processed/minute/` (1분 요약, 장비 20대) |
| 목적 | 정답(고장 전 여부)을 주고 학습하면 아이솔레이션 포레스트(AUC 0.57)보다 나아지는지 확인 |
| 상태 | 초안 (기준선 확인용, 튜닝 없음) |

## 1. 방법

- **라벨**: TTF 3개 중 하나라도 86,400초(24시간) 이하이면 고장 전(1), 아니면 정상(0).
- **입력 3가지 비교**
  - A. 센서 16개 × (평균·최소·최대) = 48열, 누적 카운터 포함
  - B. A에서 누적 카운터 3개를 뺀 39열
  - C. B + 최근 60분(행 기준) 평균·표준편차 26열 = 65열
- **학습**: 학습 장비에서 고장 전·정상을 합쳐 무작위 30만 행. `class_weight="balanced_subsample"`, 트리 100개, 깊이 14, 잎 최소 100행.
- **검증**: 장비 20대를 4묶음(5대씩)으로 나눠, 묶음마다 학습에 쓰지 않은 장비 전체 행으로 평가. 아이솔레이션 포레스트와 같은 묶음.
- **중요도**: 지니 중요도(센서별 합계)와, 한 묶음에서 순열 중요도를 함께 본다.

In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "models" else Path.cwd()
TTF = [
    "TTF_FlowCool Pressure Dropped Below Limit",
    "TTF_Flowcool Pressure Too High Check Flowcool Pump",
    "TTF_Flowcool leak",
]
COUNTERS = ["ETCHSOURCEUSAGE", "ETCHAUXSOURCETIMER", "ETCHAUX2SOURCETIMER"]
SENSORS = [
    "IONGAUGEPRESSURE", "ETCHBEAMVOLTAGE", "ETCHBEAMCURRENT",
    "ETCHSUPPRESSORVOLTAGE", "ETCHSUPPRESSORCURRENT",
    "FLOWCOOLFLOWRATE", "FLOWCOOLPRESSURE",
    "ETCHGASCHANNEL1READBACK", "ETCHPBNGASREADBACK",
    "FIXTURETILTANGLE", "ROTATIONSPEED", "ACTUALROTATIONANGLE",
    *COUNTERS, "ACTUALSTEPDURATION",
]
NOCNT = [s for s in SENSORS if s not in COUNTERS]
STATS = ["mean", "min", "max"]
ROLL = 60
TRAIN_SAMPLE = 300_000
SEED = 0

def show(fig):
    fig.update_layout(template="plotly_white", font=dict(family="Malgun Gothic, sans-serif"))
    fig.show()

## 2. 데이터 불러오기

장비별로 불러와 최근 60행(분 단위 기록 기준) 평균·표준편차를 만든다. 기록이 끊긴 구간이 있어 실제로는 60분보다 길 수 있다.

In [2]:
base_cols = [f"{s}_{st}" for s in SENSORS for st in STATS]
frames = []
for f in sorted(glob.glob(str(ROOT / "data/processed/minute/*.parquet"))):
    d = pd.read_parquet(f, columns=["Tool", *TTF, *base_cols])
    y = (d[TTF] <= 86400).any(axis=1).astype("int8")
    roll = d[[f"{s}_mean" for s in NOCNT]].rolling(ROLL, min_periods=10)
    rm = roll.mean().add_suffix("_rmean60").astype("float32")
    rs = roll.std().add_suffix("_rstd60").astype("float32")
    frames.append(pd.concat([d[["Tool"]], d[base_cols], rm, rs, y.rename("y")], axis=1))
df = pd.concat(frames, ignore_index=True)
tools = sorted(df["Tool"].unique())

feat_A = base_cols
feat_B = [f"{s}_{st}" for s in NOCNT for st in STATS]
feat_C = feat_B + [f"{s}_mean_rmean60" for s in NOCNT] + [f"{s}_mean_rstd60" for s in NOCNT]

pd.DataFrame({
    "전체 분": [f"{len(df):,}"],
    "고장 전 비율": [f"{df['y'].mean():.1%}"],
    "입력 A": [len(feat_A)], "입력 B": [len(feat_B)], "입력 C": [len(feat_C)],
}, index=["1분 요약"])

,전체 분,고장 전 비율,입력 A,입력 B,입력 C
1분 요약,"5,777,042",7.8%,48,39,65


## 3. 장비 4묶음 (아이솔레이션 포레스트와 동일)

In [3]:
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(tools))
folds = [[tools[i] for i in perm[k::4]] for k in range(4)]
pd.DataFrame({"검증 장비": [", ".join(sorted(f)) for f in folds]}, index=[f"묶음 {k+1}" for k in range(4)])

,검증 장비
묶음 1,"03M01, 04M02, 06M01, 07M02, 08M01"
묶음 2,"03M02, 05M01, 05M02, 09M01, 10M02"
묶음 3,"01M01, 01M02, 02M02, 04M01, 10M01"
묶음 4,"02M01, 06M02, 07M01, 08M02, 09M02"


## 4. 학습과 평가

In [4]:
def make_rf(k):
    return RandomForestClassifier(
        n_estimators=100, max_depth=14, min_samples_leaf=100, max_features="sqrt",
        class_weight="balanced_subsample", n_jobs=-1, random_state=k)

def run(feats, perm_fold=None):
    aucs, per_tool, imp = [], {}, []
    perm_imp = None
    for k, test_tools in enumerate(folds):
        is_test = df["Tool"].isin(test_tools).to_numpy()
        train = df[~is_test].sample(TRAIN_SAMPLE, random_state=k)
        model = make_rf(k).fit(train[feats].fillna(0), train["y"])
        test = df[is_test]
        Xt = test[feats].fillna(0)
        score = model.predict_proba(Xt)[:, 1]
        aucs.append(roc_auc_score(test["y"], score))
        for t in test_tools:
            m = (test["Tool"] == t).to_numpy()
            if test.loc[m, "y"].nunique() == 2:
                per_tool[t] = roc_auc_score(test.loc[m, "y"], score[m])
        imp.append(pd.Series(model.feature_importances_, index=feats))
        if perm_fold == k:
            sub = test.sample(50_000, random_state=0)
            r = permutation_importance(model, sub[feats].fillna(0), sub["y"], scoring="roc_auc",
                                       n_repeats=3, random_state=0, n_jobs=1)
            perm_imp = pd.Series(r.importances_mean, index=feats)
    return aucs, pd.Series(per_tool).sort_index(), pd.concat(imp, axis=1).mean(axis=1), perm_imp

res = {
    "A 카운터 포함 (48열)": run(feat_A),
    "B 카운터 제외 (39열)": run(feat_B, perm_fold=0),
    "C B + 최근 60분 (65열)": run(feat_C),
}

### 4-1. 묶음별 AUC

In [5]:
rows = []
for name, (aucs, per_tool, _, _) in res.items():
    rows.append({
        "입력": name,
        **{f"묶음 {k+1}": round(a, 3) for k, a in enumerate(aucs)},
        "평균": round(float(np.mean(aucs)), 3),
        "장비별 중앙값": round(float(per_tool.median()), 3),
        "AUC>0.5 장비": f"{int((per_tool > .5).sum())}/{len(per_tool)}",
    })
rows.append({"입력": "(참고) 아이솔레이션 포레스트 B", "묶음 1": 0.593, "묶음 2": 0.522, "묶음 3": 0.614, "묶음 4": 0.559,
             "평균": 0.572, "장비별 중앙값": 0.535, "AUC>0.5 장비": "18/20"})
pd.DataFrame(rows).set_index("입력")

,묶음 1,묶음 2,묶음 3,묶음 4,평균,장비별 중앙값,AUC>0.5 장비
입력,,,,,,,
A 카운터 포함 (48열),0.432,0.499,0.516,0.420,0.467,0.533,13/20
B 카운터 제외 (39열),0.363,0.636,0.499,0.412,0.478,0.532,16/20
C B + 최근 60분 (65열),0.415,0.681,0.512,0.443,0.513,0.553,17/20
(참고) 아이솔레이션 포레스트 B,0.593,0.522,0.614,0.559,0.572,0.535,18/20


### 4-2. 장비별 AUC

In [6]:
order = res["B 카운터 제외 (39열)"][1].sort_values().index
fig = go.Figure()
for name, (_, per_tool, _, _) in res.items():
    fig.add_bar(x=list(order), y=[per_tool.get(t, np.nan) for t in order], name=name)
fig.add_hline(y=0.5, line_dash="dash", annotation_text="무작위(0.5)")
fig.update_layout(title="장비별 AUC (학습에 쓰지 않은 장비)", xaxis_title="장비", yaxis_title="AUC", barmode="group")
show(fig)

### 4-3. 센서별 중요도

- **지니 중요도**: 입력 B와 C 각각의 4묶음 평균을, 센서별로 합친 값(통계 3종·롤링 포함).
- **순열 중요도**: 입력 B, 묶음 1의 검증 장비에서 열을 섞었을 때 AUC가 얼마나 떨어지는지를 센서별로 합친 값.

In [7]:
def by_sensor(s):
    out = {}
    for name, v in s.items():
        key = next(x for x in SENSORS if name.startswith(x + "_"))
        out[key] = out.get(key, 0) + v
    return pd.Series(out)

gB = by_sensor(res["B 카운터 제외 (39열)"][2])
gC = by_sensor(res["C B + 최근 60분 (65열)"][2])
pB = by_sensor(res["B 카운터 제외 (39열)"][3])
tab = pd.DataFrame({"지니 B": gB, "지니 C": gC, "순열 B(AUC 감소)": pB}).sort_values("지니 B", ascending=False)
tab.round(4)

,지니 B,지니 C,순열 B(AUC 감소)
ACTUALROTATIONANGLE,0.2700,0.2516,-0.0636
ROTATIONSPEED,0.1681,0.1817,-0.0723
FLOWCOOLPRESSURE,0.1509,0.1653,-0.0124
FLOWCOOLFLOWRATE,0.0587,0.0581,0.0016
IONGAUGEPRESSURE,0.0565,0.0568,-0.0087
ETCHPBNGASREADBACK,0.0542,0.0527,0.0032
ETCHBEAMCURRENT,0.0456,0.0415,0.0180
ETCHSUPPRESSORVOLTAGE,0.0443,0.0370,0.0073
ETCHSUPPRESSORCURRENT,0.0423,0.0452,-0.0056
ETCHBEAMVOLTAGE,0.0413,0.0362,0.0043


In [8]:
t = tab.iloc[::-1]
fig = go.Figure()
fig.add_bar(y=t.index, x=t["지니 B"], orientation="h", name="지니 중요도 (B)")
fig.add_bar(y=t.index, x=t["순열 B(AUC 감소)"], orientation="h", name="순열 중요도 (B, AUC 감소)")
fig.update_layout(title="센서별 중요도", barmode="group", height=620, xaxis_title="중요도")
show(fig)

## 5. 해석과 한계

- 랜덤포레스트의 장비 단위 검증 AUC는 평균 0.47~0.51로 무작위(0.5) 수준이다. 아이솔레이션 포레스트(0.57)보다도 낮다. 정답을 주고 학습해도 **처음 보는 장비에서는 고장 전 구간을 구분하지 못했다.**
- 묶음별 편차가 크다(0.36~0.68). 묶음 1·4는 0.5 아래라서, 학습 장비에서 배운 규칙이 검증 장비에서는 반대로 작동한 것이다.
- 지니 중요도 상위인 `ACTUALROTATIONANGLE`, `ROTATIONSPEED`는 검증 장비에서 열을 섞으면 오히려 AUC가 오른다(순열 중요도 음수). 학습 장비의 특성(장비·레시피 지문)을 외운 열이고, 일반화되는 신호가 아니다. **지니 중요도만으로 핵심 컬럼을 고르면 안 된다는 사례다.**
- 최근 60분 통계를 더한 입력 C가 가장 높지만(0.513) 차이는 작고 여전히 무작위 수준이다.
- 한계: 레시피·단계 정보를 쓰지 않았고, 학습 행은 무작위 30만 개라 인접 분이 많아 독립 표본이 적다. 순열 중요도는 한 묶음(5대)만 계산해 불안정하다.
- 다음 후보: 센서 값을 장비·레시피별 기준선에서 얼마나 벗어났는지(z점수)로 바꿔서 입력, 고장 종류별 모델, 라벨 창(24시간)을 짧게(1~6시간) 바꿔서 비교.